# Hybrid GCMC Analysis

Loads a single `.npz` output from `run_gcmc_hybrid.py` and checks:
1. **MD equilibration** — energy along each MD segment (requires `--dump_md_energies`)
2. **MCMC equilibration** — N(t) and E(t) stationarity
3. **Particle number statistics** — ⟨N⟩, ⟨φ⟩, distributions
4. **Acceptance rates**
5. **Configuration snapshots**

## 0 — Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Circle
from scipy.stats import mannwhitneyu
from pathlib import Path

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

In [ ]:
# ── Point this at your output file ──────────────────────────────────────────
RUN_FILE = './data/gcmc_runs/oa85.00deg_EAB5.000_mu-2.0000_kT1.0_r000/run.npz'

data = np.load(RUN_FILE, allow_pickle=False)
print('Keys:', sorted(data.keys()))

In [ ]:
# ── Unpack scalars ────────────────────────────────────────────────────────────
opening_angle = float(data['opening_angle'])
kT            = float(data['kT'])
E_AB          = float(data['E_AB'])
mu            = float(data['mu'])
box_area      = float(data['box_area'])
n_equil       = int(data['n_equil'])
n_prod        = int(data['n_prod'])
md_steps      = int(data['md_steps'])
md_dt         = float(data['md_dt'])
seed          = int(data['seed'])
r_center      = 1.0          # hard-coded in the model
L             = float(np.sqrt(box_area))
r_patch       = 1.0

# ── Trajectories ──────────────────────────────────────────────────────────────
N_traj          = data['N_traj']           # (n_prod,) int32
E_traj          = data['E_traj']           # (n_snapshots,) float32
snapshot_sweeps = data['snapshot_sweeps']  # (n_snapshots,) int32
snapshot_q      = data['snapshot_q']       # (n_snapshots, max_N, 3) float32, NaN-padded
snapshot_N      = data['snapshot_N']       # (n_snapshots,) int32
q_final         = data['q_final']          # (N_final, 3) float64

acc_md  = float(data['acc_md'])
acc_ins = float(data['acc_ins'])
acc_del = float(data['acc_del'])

HAS_MD_ENERGIES = 'md_E_traj' in data.files

phi_scale = np.pi * r_center**2 / box_area  # phi = N * phi_scale
phi_traj  = N_traj * phi_scale

print(f'opening_angle = {np.degrees(opening_angle):.2f}°   E_AB={E_AB}   kT={kT}   mu={mu}')
print(f'box: L={L:.2f}  A={box_area:.0f}')
print(f'n_equil={n_equil}  n_prod={n_prod}  seed={seed}')
print(f'MD: {md_steps} steps  dt={md_dt}  time={md_steps*md_dt:.1f} units')
print(f'snapshots: {len(snapshot_sweeps)}  (every {snapshot_sweeps[1]-snapshot_sweeps[0] if len(snapshot_sweeps)>1 else "?"} sweeps)')
print(f'MD energies available: {HAS_MD_ENERGIES}')

---
## 1 — MD Equilibration

Only available when the run was made with `--dump_md_energies`.

Plot the **mean energy per checkpoint** averaged over all MD displacement moves.  
If energy has not flattened by the last checkpoint → increase `--md_steps`.

In [ ]:
if not HAS_MD_ENERGIES:
    print('md_E_traj not present.\n'
          'Re-run with --dump_md_energies to enable this diagnostic.')
else:
    md_E = data['md_E_traj']   # (n_disp_moves, n_chunks) float32
    n_chunks = md_E.shape[1]
    chunk_steps = md_steps // n_chunks
    chunk_times = (np.arange(1, n_chunks + 1) * chunk_steps) * md_dt

    mean_E = md_E.mean(axis=0)
    std_E  = md_E.std(axis=0)
    sem_E  = std_E / np.sqrt(len(md_E))

    # Normalise: subtract mean of last quarter to see relaxation clearly
    baseline = mean_E[3 * n_chunks // 4:].mean()
    dE = mean_E - baseline

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    # Left: absolute energy
    ax = axes[0]
    ax.fill_between(chunk_times, mean_E - sem_E, mean_E + sem_E, alpha=0.25, color='steelblue')
    ax.plot(chunk_times, mean_E, color='steelblue', lw=1.5)
    ax.set_xlabel('MD time (steps × dt)')
    ax.set_ylabel('Mean total energy E')
    ax.set_title(f'MD energy profile  ({len(md_E):,} moves)')

    # Right: energy relative to plateau — shows how fast it equilibrates
    ax = axes[1]
    ax.fill_between(chunk_times, dE - sem_E, dE + sem_E, alpha=0.25, color='darkorange')
    ax.plot(chunk_times, dE, color='darkorange', lw=1.5)
    ax.axhline(0, color='k', ls='--', lw=0.8)
    ax.set_xlabel('MD time (steps × dt)')
    ax.set_ylabel('ΔE relative to plateau')
    ax.set_title('Relaxation toward equilibrium')

    plt.suptitle(f'MD equilibration check  (md_steps={md_steps}, dt={md_dt})', fontsize=12)
    plt.tight_layout()
    plt.show()

    # Quantitative: fraction of relaxation done by halfway
    if abs(dE[0]) > 1e-6:
        half = n_chunks // 2
        relaxed = 1.0 - abs(dE[half]) / abs(dE[0])
        print(f'Fraction relaxed by midpoint ({chunk_times[half]:.1f} time units): {relaxed:.1%}')
        if relaxed < 0.8:
            print('  ⚠  Less than 80% relaxed at midpoint — consider increasing --md_steps')
        else:
            print('  ✓  Energy has substantially relaxed within the MD window')

---
## 2 — MCMC Equilibration

Stationarity test on N(t) over the production window.  
Compare the first 25% vs last 50% using a Mann-Whitney U test.

In [ ]:
def running_mean(x, window):
    """Centered running mean (edge effects handled by 'valid' mode + padding)."""
    kernel = np.ones(window) / window
    return np.convolve(x, kernel, mode='same')

def is_equilibrated(traj, alpha=0.05):
    n = len(traj)
    if n < 8:
        return True, 1.0
    first = traj[:n // 4]
    last  = traj[n // 2:]
    _, p = mannwhitneyu(first, last, alternative='two-sided')
    return bool(p >= alpha), float(p)

sweeps = np.arange(1, n_prod + 1)
window = max(1, n_prod // 50)   # ~2% of trajectory for smoothing
N_smooth = running_mean(N_traj.astype(float), window)

eq, p_val = is_equilibrated(N_traj)
status    = 'EQUILIBRATED' if eq else 'NOT EQUILIBRATED'
color_stat = 'green' if eq else 'red'

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

# Top: N(t)
ax = axes[0]
ax.plot(sweeps, N_traj, lw=0.5, color='steelblue', alpha=0.6, label='N(t)')
ax.plot(sweeps, N_smooth, lw=2.0, color='navy', label=f'Running mean (w={window})')
ax.axhline(np.mean(N_traj), color='red', ls='--', lw=1.2,
           label=f'⟨N⟩ = {np.mean(N_traj):.2f} ± {np.std(N_traj):.2f}')
# shade first quarter
ax.axvspan(1, n_prod // 4, alpha=0.08, color='orange', label='First 25% (equil. check)')
ax.set_ylabel('N particles')
ax.legend(fontsize=8, loc='upper right')
ax.set_title(
    f'N(t) — Mann-Whitney p={p_val:.4f}  → {status}',
    color=color_stat, fontweight='bold'
)

# Bottom: phi(t)
ax = axes[1]
ax.plot(sweeps, phi_traj, lw=0.5, color='darkorange', alpha=0.6, label='φ(t)')
ax.plot(sweeps, running_mean(phi_traj, window), lw=2.0, color='saddlebrown')
ax.axhline(np.mean(phi_traj), color='red', ls='--', lw=1.2,
           label=f'⟨φ⟩ = {np.mean(phi_traj):.4f} ± {np.std(phi_traj):.4f}')
ax.axvspan(1, n_prod // 4, alpha=0.08, color='orange')
ax.set_xlabel('Production sweep')
ax.set_ylabel('Area fraction φ')
ax.legend(fontsize=8, loc='upper right')

plt.suptitle(f'MCMC equilibration  |  mu={mu:.3f}  kT={kT}  E_AB={E_AB}', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Also check E/N(t) at snapshot times
E_per_N = E_traj / np.maximum(snapshot_N, 1).astype(np.float32)
eq_E, p_E = is_equilibrated(E_per_N)

fig, ax = plt.subplots(figsize=(12, 3.5))
ax.plot(snapshot_sweeps, E_traj, lw=0.8, color='darkorange', alpha=0.7, label='E(t)')
ax.plot(snapshot_sweeps, running_mean(E_traj, max(1, len(E_traj)//50)),
        lw=2, color='saddlebrown', label='Running mean')
ax.axvspan(snapshot_sweeps[0], snapshot_sweeps[len(snapshot_sweeps)//4],
           alpha=0.08, color='orange', label='First 25%')
ax.set_xlabel('Production sweep')
ax.set_ylabel('Total energy E')
ax.legend(fontsize=8)
ax.set_title(
    f'Energy trajectory  (Mann-Whitney p={p_E:.4f} → {"OK" if eq_E else "NOT EQUILIBRATED"})',
    color='green' if eq_E else 'red', fontweight='bold'
)
plt.tight_layout()
plt.show()

---
## 3 — Particle Number Statistics

In [ ]:
mean_N   = np.mean(N_traj)
std_N    = np.std(N_traj)
mean_phi = np.mean(phi_traj)
std_phi  = np.std(phi_traj)

# Use last 50% as "equilibrated" production window
N_prod_eq   = N_traj[n_prod // 2:]
phi_prod_eq = N_prod_eq * phi_scale

print(f'Full production window:')
print(f'  ⟨N⟩  = {mean_N:.2f} ± {std_N:.2f}')
print(f'  ⟨φ⟩  = {mean_phi:.4f} ± {std_phi:.4f}')
print(f'  N min/max = {N_traj.min()} / {N_traj.max()}')
print()
print(f'Last 50% (post-equilibration):')
print(f'  ⟨N⟩  = {np.mean(N_prod_eq):.2f} ± {np.std(N_prod_eq):.2f}')
print(f'  ⟨φ⟩  = {np.mean(phi_prod_eq):.4f} ± {np.std(phi_prod_eq):.4f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# N histogram
ax = axes[0]
bins = np.arange(N_traj.min() - 0.5, N_traj.max() + 1.5)
ax.hist(N_traj,     bins=bins, alpha=0.4, color='steelblue', label='All production')
ax.hist(N_prod_eq,  bins=bins, alpha=0.6, color='navy',      label='Last 50%')
ax.axvline(mean_N, color='red', ls='--', lw=1.5, label=f'⟨N⟩={mean_N:.1f}')
ax.set_xlabel('N particles')
ax.set_ylabel('Count')
ax.set_title('N distribution')
ax.legend(fontsize=8)

# phi histogram
ax = axes[1]
phi_bins = np.linspace(phi_traj.min() - 0.005, phi_traj.max() + 0.005, 40)
ax.hist(phi_traj,    bins=phi_bins, alpha=0.4, color='darkorange', label='All production')
ax.hist(phi_prod_eq, bins=phi_bins, alpha=0.6, color='saddlebrown', label='Last 50%')
ax.axvline(mean_phi, color='red', ls='--', lw=1.5, label=f'⟨φ⟩={mean_phi:.4f}')
ax.set_xlabel('Area fraction φ')
ax.set_ylabel('Count')
ax.set_title('φ distribution')
ax.legend(fontsize=8)

plt.suptitle(f'Particle number statistics  |  mu={mu:.3f}', fontsize=12)
plt.tight_layout()
plt.show()

---
## 4 — Autocorrelation of N(t)

The autocorrelation time τ tells you how many sweeps between statistically independent samples.

In [ ]:
def autocorr(x, max_lag=None):
    """Normalised autocorrelation function via FFT."""
    x = x - x.mean()
    n = len(x)
    if max_lag is None:
        max_lag = n // 4
    f   = np.fft.rfft(x, n=2 * n)
    acf = np.fft.irfft(f * np.conj(f))[:n]
    acf /= acf[0]
    return acf[:max_lag]

max_lag = min(n_prod // 4, 2000)
acf_N   = autocorr(N_traj.astype(float), max_lag=max_lag)
lags    = np.arange(max_lag)

# Integrated autocorrelation time (up to first zero crossing)
zero_cross = np.where(acf_N < 0)[0]
tau_int = float(0.5 + np.sum(acf_N[1: zero_cross[0] if len(zero_cross) else max_lag]))

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(lags, acf_N, lw=1.2, color='steelblue')
ax.axhline(0,   color='k',   lw=0.8, ls='--')
ax.axhline(1/np.e, color='orange', lw=1, ls=':', label=f'1/e  (τ₁/ₑ ≈ {lags[np.argmin(np.abs(acf_N - 1/np.e))]:.0f} sweeps)')
ax.axvline(tau_int, color='red', lw=1.5, ls='--', label=f'τ_int ≈ {tau_int:.1f} sweeps')
ax.set_xlabel('Lag (sweeps)')
ax.set_ylabel('ACF(N)')
ax.set_title('Autocorrelation of N(t)')
ax.set_xlim(0, max_lag)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

n_eff = n_prod / (2 * tau_int)
print(f'Integrated autocorrelation time: τ_int ≈ {tau_int:.1f} sweeps')
print(f'Effective sample size:           n_eff ≈ {n_eff:.0f}')
print(f'SEM(⟨N⟩) accounting for τ_int:  {std_N / np.sqrt(n_eff):.3f}')

---
## 5 — Acceptance Rates

In [ ]:
labels = ['Displacement\n(MD, always 1)', 'Insertion', 'Deletion']
rates  = [acc_md, acc_ins, acc_del]
colors = ['steelblue', 'seagreen', 'tomato']

fig, ax = plt.subplots(figsize=(6, 3.5))
bars = ax.bar(labels, rates, color=colors, width=0.5, edgecolor='k', linewidth=0.6)
for bar, rate in zip(bars, rates):
    ax.text(bar.get_x() + bar.get_width()/2, rate + 0.01,
            f'{rate:.4f}', ha='center', va='bottom', fontsize=10)
ax.set_ylim(0, 1.15)
ax.set_ylabel('Acceptance fraction')
ax.set_title('GCMC acceptance rates')
ax.grid(axis='y', alpha=0.3)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

print(f'Displacement: {acc_md:.4f}  (always 1.0 for Langevin MD)')
print(f'Insertion:    {acc_ins:.4f}')
print(f'Deletion:     {acc_del:.4f}')
if acc_ins < 1e-3:
    print('  ⚠  Very low insertion rate — mu may be too low (too sparse) or'
          ' E_AB too strong (hard to insert into assembled clusters)')
if acc_del < 1e-3:
    print('  ⚠  Very low deletion rate — mu may be too high (too dense)')

---
## 6 — Configuration Snapshots

Evenly-spaced frames from the production trajectory.

In [ ]:
def plot_config(ax, q, L, params_oa, r_patch=1.0, r_c=1.0, title=''):
    """Draw one frame.  q: (N, 3) [x, y, theta]."""
    ax.set_xlim(-L/2, L/2)
    ax.set_ylim(-L/2, L/2)
    ax.set_aspect('equal')
    ax.set_xticks([]); ax.set_yticks([])
    ax.add_patch(plt.Rectangle((-L/2, -L/2), L, L,
                               fill=False, edgecolor='dimgray', lw=1.0))
    if len(q) == 0:
        ax.text(0, 0, 'N=0', ha='center', va='center', fontsize=9, color='gray')
        ax.set_title(title, fontsize=7)
        return
    for xi, yi, _ in q:
        ax.add_patch(Circle((xi, yi), r_c, color='steelblue', alpha=0.35, lw=0))
    oa  = params_oa
    th  = q[:, 2]
    c   = q[:, :2]
    pA  = c + r_patch * np.stack([np.cos(th),      np.sin(th)],      axis=-1)
    pB  = c + r_patch * np.stack([np.cos(th + oa), np.sin(th + oa)], axis=-1)
    ax.scatter(pA[:, 0], pA[:, 1], s=15, c='red',   zorder=3, linewidths=0)
    ax.scatter(pB[:, 0], pB[:, 1], s=15, c='limegreen', zorder=3, linewidths=0)
    ax.set_title(f'{title}  N={len(q)}', fontsize=7, pad=2)


n_frames = min(12, len(snapshot_sweeps))    # show up to 12 frames
frame_idx = np.round(np.linspace(0, len(snapshot_sweeps) - 1, n_frames)).astype(int)

ncols = 4
nrows = int(np.ceil(n_frames / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3.5, nrows * 3.5))
axes = np.array(axes).flatten()

for k, fi in enumerate(frame_idx):
    Ni = int(snapshot_N[fi])
    q  = snapshot_q[fi, :Ni, :].astype(np.float64)   # unpad
    plot_config(axes[k], q, L, opening_angle, r_patch=r_patch, r_c=1.0,
                title=f'sweep {snapshot_sweeps[fi]}')

for ax in axes[n_frames:]:
    ax.set_visible(False)

plt.suptitle(
    f'Snapshots  |  opening={np.degrees(opening_angle):.1f}°  '
    f'E_AB={E_AB}  mu={mu:.2f}',
    fontsize=11, y=1.01
)
plt.tight_layout()
plt.show()

---
## 7 — Energy per Particle vs Time

In [ ]:
E_per_N = E_traj / np.maximum(snapshot_N, 1).astype(np.float32)
smooth_w = max(1, len(E_per_N) // 30)

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

ax = axes[0]
ax.plot(snapshot_sweeps, E_traj, lw=0.6, alpha=0.6, color='darkorange')
ax.plot(snapshot_sweeps, running_mean(E_traj, smooth_w), lw=2, color='saddlebrown',
        label='Running mean')
ax.set_ylabel('Total energy E')
ax.legend(fontsize=8)
ax.set_title('Total energy')

ax = axes[1]
ax.plot(snapshot_sweeps, E_per_N, lw=0.6, alpha=0.6, color='purple')
ax.plot(snapshot_sweeps, running_mean(E_per_N, smooth_w), lw=2, color='indigo',
        label='Running mean')
ax.axhline(np.mean(E_per_N), color='red', ls='--', lw=1.2,
           label=f'⟨E/N⟩ = {np.mean(E_per_N):.3f}')
ax.set_xlabel('Production sweep')
ax.set_ylabel('Energy per particle E/N')
ax.legend(fontsize=8)
ax.set_title('Energy per particle')

plt.suptitle('Energy trajectory', fontsize=12)
plt.tight_layout()
plt.show()

---
## 8 — Summary

In [ ]:
eq_N, p_N = is_equilibrated(N_traj)
eq_E, p_E = is_equilibrated(E_per_N)

lines = [
    '=' * 55,
    'SIMULATION SUMMARY',
    '=' * 55,
    f'  opening_angle  = {np.degrees(opening_angle):.2f}°',
    f'  E_AB           = {E_AB}',
    f'  mu             = {mu:.4f}',
    f'  kT             = {kT}',
    f'  box_area       = {box_area:.0f}   L = {L:.2f}',
    f'  n_equil        = {n_equil}',
    f'  n_prod         = {n_prod}',
    f'  MD: {md_steps} steps  dt={md_dt}  ({md_steps*md_dt:.1f} time units)',
    '-' * 55,
    f'  ⟨N⟩            = {np.mean(N_traj):.2f} ± {np.std(N_traj):.2f}',
    f'  ⟨φ⟩            = {np.mean(phi_traj):.4f} ± {np.std(phi_traj):.4f}',
    f'  ⟨E/N⟩          = {np.mean(E_per_N):.3f} ± {np.std(E_per_N):.3f}',
    f'  acc_ins        = {acc_ins:.4f}',
    f'  acc_del        = {acc_del:.4f}',
    f'  τ_int(N)       ≈ {tau_int:.1f} sweeps',
    f'  n_eff          ≈ {n_eff:.0f}',
    '-' * 55,
    f'  N equilibrated?  {"YES" if eq_N else "NO"}   (p={p_N:.4f})',
    f'  E equilibrated?  {"YES" if eq_E else "NO"}   (p={p_E:.4f})',
]
if HAS_MD_ENERGIES:
    md_E_loc = data['md_E_traj']
    lines.append(f'  MD E (first→last chunk): {md_E_loc[:,0].mean():.2f} → {md_E_loc[:,-1].mean():.2f}')

lines.append('=' * 55)
print('\n'.join(lines))